# 🖼️ Image Recognition with Pre-trained MobileNetV2

Welcome! In this project, we are going to explore **Computer Vision** and **Deep Learning** without training a massive model from scratch. 
Instead, we will use a concept called **transfer learning**. We'll load **MobileNetV2** (designed by Google), which has already been trained on **ImageNet** (a dataset of over 1.2 million images across 1,000 object categories). It already knows how to recognize cats, dogs, cars, pizza, and much more! 🚀✨

## 1. ⚙️ Setup & Imports

Let's import TensorFlow/Keras, NumPy, Pillow (for image processing), and matplotlib (for viewing images).

In [ ]:
import os
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

print("✅ Setup successfully loaded!")

## 2. 📊 Load ImageNet Classes

ImageNet contains 1,000 distinct categories. Let's load the class index map so we can explore the labels. We'll download the index if it isn't available locally.

In [ ]:
class_url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
class_path = "../dataset/imagenet_class_index.json"

if not os.path.exists(class_path):
    print("📥 Downloading ImageNet class index...")
    os.makedirs(os.path.dirname(class_path), exist_ok=True)
    urllib.request.urlretrieve(class_url, class_path)
    print("✅ Download complete!")

with open(class_path, "r") as f:
    class_index = json.load(f)

print(f"📦 Loaded {len(class_index)} categories!")
print("Sample categories:")
for i in range(5):
    print(f"Class {i}: {class_index[str(i)][1]}")

## 3. 🧠 Load MobileNetV2 Model

We load Google's MobileNetV2 weights trained on the ImageNet dataset. The model has about 2.2 million parameters but only occupies ~14MB of space, making it perfect for real-time mobile and web deployment!

In [ ]:
print("🏋️ Loading MobileNetV2 model...")
model = MobileNetV2(weights='imagenet')
print("✅ Model loaded successfully!")
model.summary()

## 4. 📷 Image Loading & Preprocessing

Neural networks expect inputs in a strict format. Before we pass an image to MobileNetV2, we need to:
1. **Load it** using Pillow.
2. **Resize it** to `224x224` pixels.
3. **Convert it to a NumPy array** and add a batch dimension (making shape `(1, 224, 224, 3)`).
4. **Normalise pixel intensities** to be scaled between `-1` and `1` using MobileNetV2's built-in `preprocess_input`.

In [ ]:
# Path to our sample dog image
img_path = "../Images/sample_dog.png"

# Open and show the image
img = Image.open(img_path)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title("Original Image")
plt.show()

# Preprocessing
# Ensure RGB (in case image has an alpha channel)
if img.mode != "RGB":
    img = img.convert("RGB")

img_resized = img.resize((224, 224))
x = np.array(img_resized, dtype=np.float32)
x = np.expand_dims(x, axis=0)
x_preprocessed = preprocess_input(x)

print(f"Image raw shape: {np.array(img).shape}")
print(f"Preprocessed input shape: {x_preprocessed.shape}")

## 5. ⚡ Model Inference & Prediction

Now we pass our preprocessed image matrix into the model to predict the category. We decode the predictions to print out the top-5 candidates along with their confidence probabilities.

In [ ]:
preds = model.predict(x_preprocessed)
decoded_preds = decode_predictions(preds, top=5)[0]

print("🎯 Top-5 Predictions:")
for rank, (imagenet_id, label, prob) in enumerate(decoded_preds):
    formatted_label = label.replace("_", " ").title()
    print(f"{rank + 1}. {formatted_label}: {prob * 100:.2f}%")